# 08 — Model training

Ports and **fixes** `notebooks_original/model-selection.ipynb` (see `PROJECT_PLAN.md` §5,
`CLAUDE.md` "highest priority" bug).

**The original bug.** That notebook ran a `GridSearchCV` that scored ~0.90 R², then
pickled a *different*, hand-rebuilt pipeline — different `n_estimators`, no
`max_depth`/`max_features`/`max_samples`, `OneHotEncoder` instead of `TargetEncoder`
for `sector` — fit on all of `X` with no held-out test. The 0.90 was never the
shipped model's number.

**What this notebook does.** All logic lives in `src/models/`; this is the thin
runner. It:

1. loads `data/processed/gurgaon_properties_post_feature_selection_v2.csv`,
2. splits off a 20% test set (`random_state=42`) that never touches the search,
3. runs `RandomizedSearchCV` (`n_iter=60`, 10-fold CV) over the RF grid with **one**
   preprocessing scheme,
4. exports `search.best_estimator_` **verbatim** to `models/price_pipeline.pkl`,
5. scores that exact object on the held-out test set.

CV R² and test metrics are reported **separately** — they are different numbers.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import numpy as np
import pandas as pd
import sklearn

from src.models.train import train_and_export, FEATURE_COLS
from src.models.predict import load_model, predict_one

print("sklearn", sklearn.__version__)
print("features:", FEATURE_COLS)

sklearn 1.9.0
features: ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room', 'sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category', 'property_type']


## Run the search + export

`train_and_export` writes three artefacts:

| Path | Contents |
|---|---|
| `models/price_pipeline.pkl` | the exported pipeline (`search.best_estimator_`) |
| `models/model_metrics.json` | CV score, test metrics, full provenance |
| `reports/model/model_selection_log.md` | human-readable decision log |


In [2]:
report = train_and_export(n_iter=60)

Fitting 10 folds for each of 60 candidates, totalling 600 fits


## The CV number (training split, comparable to the original's `best_score_`)

In [3]:
s = report["search"]
print(f'RandomizedSearchCV  n_iter={s["n_iter"]}  candidates={s["n_candidates_evaluated"]}')
print(f'CV R2 (log target): {s["best_cv_r2_mean"]:.4f} +/- {s["best_cv_r2_std"]:.4f}')
print("selected hyper-parameters:")
for k, v in s["best_params"].items():
    print(f'  {k} = {v}')

RandomizedSearchCV  n_iter=60  candidates=60
CV R2 (log target): 0.8811 +/- 0.0161
selected hyper-parameters:
  regressor__n_estimators = 500
  regressor__max_samples = 1.0
  regressor__max_features = sqrt
  regressor__max_depth = 20


## The test number (held-out split, for the *actual exported* model)

In [4]:
em = report["exported_model"]
print("provenance:", em["provenance"])
print()
print(f'{"metric":<14}{"test":>14}{"train":>14}')
for key, label in [("r2_log", "R2 (log)"), ("r2_crore", "R2 (crore)"),
                   ("mae_crore", "MAE (Cr)"), ("rmse_crore", "RMSE (Cr)")]:
    print(f'{label:<14}{em["test_metrics"][key]:>14.4f}{em["train_metrics"][key]:>14.4f}')

provenance: search.best_estimator_, refit by RandomizedSearchCV on X_train only; exported without modification

metric                  test         train
R2 (log)              0.9102        0.9752
R2 (crore)            0.8514        0.9523
MAE (Cr)              0.4913        0.2555
RMSE (Cr)             1.0686        0.6088


In [5]:
print(json.dumps(report, indent=2))

{
  "created_utc": "2026-09-08T10:20:11+00:00",
  "random_state": 42,
  "sklearn_version": "1.9.0",
  "data_file": "data/processed/gurgaon_properties_post_feature_selection_v2.csv",
  "data_shape": [
    3554,
    13
  ],
  "n_train": 2843,
  "n_test": 711,
  "target_transform": "log1p(price)",
  "feature_columns": [
    "bedRoom",
    "bathroom",
    "built_up_area",
    "servant room",
    "store room",
    "sector",
    "balcony",
    "agePossession",
    "furnishing_type",
    "luxury_category",
    "floor_category",
    "property_type"
  ],
  "encoding": {
    "numeric": {
      "cols": [
        "bedRoom",
        "bathroom",
        "built_up_area",
        "servant room",
        "store room"
      ],
      "encoder": "StandardScaler"
    },
    "sector": {
      "encoder": "TargetEncoder",
      "internal_cv": true,
      "n_categories": 104
    },
    "ordinal": {
      "cols": [
        "balcony",
        "agePossession",
        "furnishing_type",
        "luxury_category",

## Sanity check — load the exported model and predict one row

In [6]:
model = load_model()
price_cr = predict_one(
    model,
    property_type="flat", sector="sector 102", bedRoom=3.0, bathroom=2.0,
    balcony="2", agePossession="New Property", built_up_area=1800.0,
    **{"servant room": 0.0, "store room": 0.0},
    furnishing_type=0.0, luxury_category="Medium", floor_category="Mid Floor",
)
print(f'predicted price: Rs {price_cr:.2f} Cr')

predicted price: Rs 1.38 Cr


## Takeaway

- **CV R² (log target)** — reported above, on the training split. This is the
  number comparable to the original notebook's `search.best_score_` (0.9027).
- **Test R² / MAE / RMSE** — reported above, for `models/price_pipeline.pkl`
  exactly as shipped. These go into `docs/model_documentation.md` later, both
  labelled, per `PROJECT_PLAN.md` §5.

Full rationale (encoding choices, leakage argument, deviations from the original):
`reports/model/model_selection_log.md`.
